In [ ]:
import market as mkt
import jointsim as js
import var as v
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
#from scipy.optimize import minimize
#from noisyopt import minimizeCompass
from skopt import gp_minimize

relpath = os.path.dirname(os.path.abspath(''))
cob_date_str = '20251231'
ptfs = ['SP500_LongAll','SP500_Hedged']
measures= [0.05, 0.01]
output_path_part = 'C:\\Temp'
figpath = 'C:\\dev\\MLCopula\\document\\figures'

market_and_wghts =[]
for ptf_name in ptfs:
    ptf_df = pd.read_csv(f'{relpath}\\data\\portfolios\\{ptf_name}.csv')
    ptf_names, ptf_wghts = list(ptf_df['Symbol']), list(ptf_df['Weight'])

    market = mkt.Market(name=ptf_name, cob_date_str=cob_date_str, ticker_list=ptf_names, period_years=7, interval='1d', cache_path=output_path_part)
    market.load_history()
    market_and_wghts.append((market,ptf_wghts))
    
cob, res_c, res_s =[], [], []
for i in range(2):
    new_cob_date = datetime.strptime(cob_date_str, '%Y%m%d').date() - timedelta(days=30*i)
    
    def error_func(x):
        error = 0.0
        for (market,ptf_wghts) in market_and_wghts:
            returns = market.get_logreturns(cob_date_str=new_cob_date.strftime('%Y%m%d'), sub_period_years=1)

            copula = js.MixCopulaNumTest(returns, [1.0]*len(ptf_names), weights=x)
            port_sim = v.VarSim(returns, market.spot, copula, np.array(ptf_wghts), method='midpoint')
            port_sim.calculate_pnls(10000)

            copula_hist = js.HistSimulation(returns, [1.0]*len(ptf_names))
            port_sim_hist = v.VarSim(returns, market.spot, copula_hist, np.array(ptf_wghts), method='midpoint')
            port_sim_hist.calculate_pnls(10000)

            for measure in measures:
                port_var = -round(port_sim.get_quantile(measure),2)
                port_var_hist = -round(port_sim_hist.get_quantile(measure),2)
                error_per_measure = abs(port_var - port_var_hist)
                error = error + error_per_measure

            
        print(f'-----> total error: {error}')
        return error
    
    #res = minimize(error_func, [0.0, 0.99], bounds= ,method='Nelder-Mead')
    #res = minimizeCompass(error_func, bounds=((0.0,1.0), (0.0,1.0)), x0=[0.9, 0.1], deltatol=0.1, paired=False)
    res = gp_minimize(error_func, [(0.0,1.0), (0.0,1.0)])

    cob.append(new_cob_date)
    res_c.append(list(res.x)[0])
    res_s.append(list(res.x)[1])


res_dict = {'COB': cob, 'CauchyWeight': res_c, 'CauchySkew':res_s}
res_full = pd.DataFrame(data=res_dict)
res_full.set_index('COB', inplace=True); res_full.sort_index(inplace=True)
res_full.to_csv(f'{output_path_part}\\OptimalCopula.csv')
res_full.plot().legend(bbox_to_anchor=(0., 1.02, 1., .102), fontsize='small').set_title(f'OptimalCopula')
plt.savefig(f'{figpath}\\OptimalCopula.png', bbox_inches="tight")
plt.show()

Market stats 20251231-1Y, non nones count: 120032, needs 120032
Simulation stats, non nones count: 4839516, needs 4839516
Simulation stats, non nones count: 4801280, needs 4801280
Market stats 20251231-1Y, non nones count: 120032, needs 120032
Simulation stats, non nones count: 4839516, needs 4839516
Simulation stats, non nones count: 4801280, needs 4801280
-----> total error: 4.47
Market stats 20251231-1Y, non nones count: 120032, needs 120032
Simulation stats, non nones count: 4839516, needs 4839516
Simulation stats, non nones count: 4801280, needs 4801280
Market stats 20251231-1Y, non nones count: 120032, needs 120032
Simulation stats, non nones count: 4839516, needs 4839516
Simulation stats, non nones count: 4801280, needs 4801280
-----> total error: 3.7999999999999985
Market stats 20251231-1Y, non nones count: 120032, needs 120032
Simulation stats, non nones count: 4839516, needs 4839516
Simulation stats, non nones count: 4801280, needs 4801280
Market stats 20251231-1Y, non nones 

c:\dev\MLCopula\.myvenv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\dev\MLCopula\.myvenv\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


Simulation stats, non nones count: 4840000, needs 4840000
Simulation stats, non nones count: 4801280, needs 4801280
Market stats 20251231-1Y, non nones count: 120032, needs 120032


c:\dev\MLCopula\.myvenv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\dev\MLCopula\.myvenv\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


Simulation stats, non nones count: 4840000, needs 4840000
Simulation stats, non nones count: 4801280, needs 4801280
-----> total error: 3.34
Market stats 20251231-1Y, non nones count: 120032, needs 120032
